# File export "bẩn" từ ERP: đọc đúng trước khi tính

Tệp `tests/fixtures/ventes_france_2011-11.csv` là **1 580 dòng thật** của Online Retail II (UCI, CC BY 4.0): mọi dòng
hoá đơn của khách Pháp trong tháng 11/2011. Văn phòng Paris xuất nó từ hệ thống ERP của họ, nên tệp mang đủ thói quen
của một máy Windows cài tiếng Pháp:

- mã hoá `cp1252` (không phải UTF-8), tiêu đề cột tiếng Pháp có dấu (`Désignation`, `Quantité`, `N° client`);
- phân cách cột bằng `;`, dấu thập phân là `,` (`2,55`);
- ngày dạng `dd/mm/yyyy HH:MM`;
- 2 dòng mở đầu trước tiêu đề, 1 dòng `Total lignes;<số dòng>` ở cuối (control total của hệ thống nguồn).

**Bối cảnh.** Trưởng nhóm kinh doanh châu Âu hỏi: "Tháng 11 khách Pháp mua bao nhiêu, bao nhiêu khách, bao nhiêu đơn bị
huỷ?" Câu trả lời chỉ đúng khi tệp được đọc đúng: sai encoding thì lỗi ngay, nhưng sai dấu thập phân hoặc thứ tự ngày
thì **không lỗi gì cả**, chỉ ra số sai.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu và in kết quả; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/05-data-collection/exercises/basic/02-messy-erp-export -q`

In [ ]:
# explore: tìm gốc repo và đường dẫn tệp export
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
EXPORT = ROOT / "tracks/data-analyst/modules/05-data-collection/exercises/basic/02-messy-erp-export/tests/fixtures/ventes_france_2011-11.csv"
raw = EXPORT.read_bytes()
print(f"{len(raw):,} byte")
print(raw[:200])

In [ ]:
import codecs
import io
from pathlib import Path

import pandas as pd

In [ ]:
# explore: ba cách đọc "nhanh" và chuyện gì xảy ra
try:
    pd.read_csv(EXPORT)
except Exception as exc:
    print("1) mặc định:", type(exc).__name__, str(exc)[:90])

naive = pd.read_csv(EXPORT, encoding="cp1252")
print("2) đúng encoding, sai delimiter:", naive.shape, "-> không lỗi, nhưng chỉ có 1 cột")

semi = pd.read_csv(EXPORT, encoding="cp1252", sep=";", skiprows=2, skipfooter=1, engine="python")
print("3) đúng delimiter, quên decimal:")
print(semi.dtypes)

## 1. Encoding và delimiter

`detect_encoding(raw: bytes) -> str`: trả `"utf-8-sig"` nếu bytes bắt đầu bằng BOM UTF-8 (`codecs.BOM_UTF8`), `"utf-8"`
nếu giải mã UTF-8 được, ngược lại `"cp1252"` (bảng mã mặc định của Excel/Windows ở Tây Âu, giải mã được gần như mọi
byte nên dùng làm phương án cuối).

`sniff_delimiter(line: str) -> str`: trong các ký tự `;`, `,`, `\t`, `|`, trả ký tự xuất hiện nhiều nhất trên **dòng
tiêu đề** `line`; không có ký tự nào thì `ValueError`.

In [ ]:
DELIMITERS = (";", ",", "\t", "|")


def detect_encoding(raw: bytes) -> str:
    """utf-8-sig nếu có BOM, utf-8 nếu giải mã được, ngược lại cp1252."""
    # TODO
    raise NotImplementedError


def sniff_delimiter(line: str) -> str:
    """Ký tự phân cách xuất hiện nhiều nhất trên dòng tiêu đề."""
    # TODO: đếm từng ký tự trong DELIMITERS; không có thì ValueError
    raise NotImplementedError

In [ ]:
# explore: encoding và delimiter của tệp thật
enc = detect_encoding(raw)
lines = raw.decode(enc).splitlines()
print("encoding:", enc)
for i, line in enumerate(lines[:4]):
    print(i, repr(line[:90]))
print("delimiter trên dòng tiêu đề:", repr(sniff_delimiter(lines[2])))
print("dòng cuối:", repr(lines[-1]))

## 2. Đọc export thành bảng chuẩn

`read_export(path) -> pd.DataFrame`:

- đọc bytes, giải mã bằng `detect_encoding`;
- tiêu đề là **dòng đầu tiên bắt đầu bằng `Facture`** (bỏ mọi dòng phía trên); bỏ dòng trống và dòng bắt đầu bằng
  `Total`;
- delimiter lấy từ `sniff_delimiter(dòng tiêu đề)`; dấu thập phân là `,` khi delimiter là `;`, ngược lại `.`;
- đổi tên cột sang tên chuẩn của bộ gốc, theo đúng thứ tự `RENAME` dưới đây;
- kiểu dữ liệu: `Invoice`, `StockCode` là chuỗi (giữ `C573868`, giữ số 0 đầu); `Quantity` số nguyên; `Price` số thực;
  `Customer ID` kiểu `Int64` (số nguyên cho phép thiếu, không thành `12674.0`); `InvoiceDate` là datetime đọc theo
  **ngày trước, tháng sau** (`format="%d/%m/%Y %H:%M"`).

`control_total(path) -> int`: số ghi trên dòng `Total lignes;<n>` ở cuối tệp.

In [ ]:
RENAME = {"Facture": "Invoice", "Code article": "StockCode", "Désignation": "Description", "Quantité": "Quantity",
          "Date facture": "InvoiceDate", "Prix unitaire": "Price", "N° client": "Customer ID", "Pays": "Country"}


def read_export(path) -> pd.DataFrame:
    """Đọc export ERP thành bảng 8 cột chuẩn với kiểu dữ liệu đúng."""
    # TODO: giải mã, tìm dòng tiêu đề, bỏ dòng Total, read_csv với sep/decimal/dtype, đổi tên, parse ngày
    raise NotImplementedError


def control_total(path) -> int:
    """Số dòng hệ thống nguồn ghi ở dòng `Total lignes`."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: đọc đúng, đối chiếu control total
sales = read_export(EXPORT)
print(sales.dtypes)
print(f"{len(sales)} dòng đọc được, control total {control_total(EXPORT)}")
sales.head()

In [ ]:
# explore: nếu đọc ngày theo kiểu Mỹ (tháng trước) thì sao?
as_us = pd.to_datetime(semi["Date facture"], format="%m/%d/%Y %H:%M", errors="coerce")
print("dòng parse 'được' nhưng sai tháng:", int(as_us.notna().sum()))
print("  trong đó trùng hợp vẫn đúng (ngày 11/11):", int((as_us.dt.month == 11).sum()))
print("dòng không parse được (ngày > 12):", int(as_us.isna().sum()))
print(as_us.dt.month.value_counts().sort_index())

## 3. Tóm tắt cho trưởng nhóm kinh doanh

`summarize(df) -> dict` với đúng các khoá:

- `rows`: số dòng; `invoices`: số hoá đơn khác nhau; `cancelled_invoices`: số hoá đơn khác nhau bắt đầu bằng `C`;
- `customers`: số `Customer ID` khác nhau (không tính thiếu);
- `valid_lines`: số dòng hợp lệ theo định nghĩa chung của track (bỏ hoá đơn `C...`, bỏ `Quantity <= 0` hoặc
  `Price <= 0`);
- `revenue`: tổng `Quantity * Price` trên dòng hợp lệ, làm tròn 2 chữ số.

In [ ]:
def summarize(df: pd.DataFrame) -> dict:
    """rows, invoices, cancelled_invoices, customers, valid_lines, revenue."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: số cho trưởng nhóm, và phí vận chuyển (StockCode POST) chiếm bao nhiêu
summary = summarize(sales)
print(summary)
valid = sales[~sales.Invoice.str.startswith("C") & (sales.Quantity > 0) & (sales.Price > 0)]
line_rev = valid.Quantity * valid.Price
post = line_rev[valid.StockCode == "POST"].sum()
print(f"POST: {int((valid.StockCode == 'POST').sum())} dòng, {post:,.2f} GBP = {post / line_rev.sum():.1%} doanh thu")
print(f"doanh thu dòng không có Customer ID: {line_rev[valid['Customer ID'].isna()].sum():,.2f} GBP")

In [ ]:
# explore (tuỳ chọn, lần đầu ~40 giây, sau đó đọc cache): đối chiếu với tệp gốc nếu đã tải online-retail-ii
import os
import tempfile


def read_sheet(xlsx: Path, sheet: str) -> pd.DataFrame:
    # đọc một sheet, cache parquet trong thư mục tạm của hệ thống (khoá theo mtime/size của tệp gốc)
    stat = xlsx.stat()
    cache = (Path(tempfile.gettempdir()) / "da-cache"
             / f"online_retail_{sheet.replace(' ', '_')}_{stat.st_mtime_ns}_{stat.st_size}.parquet")
    if cache.exists():
        return pd.read_parquet(cache)
    df = pd.read_excel(xlsx, sheet_name=sheet, dtype={"Invoice": str, "StockCode": str, "Description": str})
    cache.parent.mkdir(parents=True, exist_ok=True)
    tmp = cache.with_name(f"{cache.name}.{os.getpid()}.tmp")
    df.to_parquet(tmp)
    os.replace(tmp, cache)
    return df

src = ROOT / "data/raw/online-retail-ii/online_retail_II.xlsx"
if src.exists():
    full = read_sheet(src, "Year 2010-2011")
    ref = full[(full.Country == "France") & (full.InvoiceDate.dt.to_period("M") == "2011-11")]
    print("số dòng nguồn:", len(ref), "| doanh thu nguồn:", summarize(ref)["revenue"])
else:
    print("Chưa tải online-retail-ii, bỏ qua đối chiếu (uv run python data/download_dataset.py online-retail-ii)")

## 4. Đọc kết quả, rồi làm trắc nghiệm

Trả lời trưởng nhóm kinh doanh trong ba câu, kèm một câu về độ tin cậy của số:

1. Doanh thu, số đơn, số khách của khách Pháp tháng 11/2011; bao nhiêu đơn bị huỷ.
2. Bao nhiêu doanh thu là phí vận chuyển (`POST`), có nên tính vào "doanh thu sản phẩm" không?
3. Bạn đã kiểm gì để chắc tệp được đọc đủ và đúng (control total, khoảng ngày, kiểu cột)?

Xong thì mở app, làm phần **trắc nghiệm** của bài.